In [ ]:
from youtube_comment_downloader import *
import pandas as pd
semidata={}
urls=[]
while True:
    category_name=input("enter the name of the categroy-:\n example\n the videos are about a specific game,product (-1 to exit)")
    if category_name.strip()=='-1':
        break
    while True:
        url=input(f"enter a url related to{category_name} (-1 to start a new category)")
        if url.strip()=='-1':
            break
        semidata[url]=category_name
        urls.append(url)

extractor=YoutubeCommentDownloader()
data=[]
for url in urls:
    comments=extractor.get_comments_from_url(url)
    for c in comments:
        data.append({
            'url':url,
            'author':c.get("author"),
            'Text': c.get("text"),
            'date': c.get("time")
            })
data=pd.DataFrame(data)
data['category']=data['url'].map(semidata)
data.to_csv("Checkpoint1")        

In [3]:
import pandas as pd
data=pd.read_csv(r"C:\Users\Prabhat\Desktop\Carear\projects\Auto snetiment analyser\data\Checkpoint1")

In [ ]:
data.drop_duplicates(inplace=True)

,Unnamed: 0,url,author,Text,date,category
0,0,https://www.youtube.com/watch?v=n1gwrRFc4Wc&pp...,@Ronak-y3z,GIVE OLD BRAWL STAR 🙏🏻,43 minutes ago,Brawl Stars
1,1,https://www.youtube.com/watch?v=n1gwrRFc4Wc&pp...,@ДамианТертышник,1:05 9999 1:09 1:11 1:11 1:11 1:12 1:12 1:12 1...,46 minutes ago,Brawl Stars
2,2,https://www.youtube.com/watch?v=n1gwrRFc4Wc&pp...,@CalebKlengenberg,I can’t play the game,50 minutes ago,Brawl Stars
3,3,https://www.youtube.com/watch?v=n1gwrRFc4Wc&pp...,@Muhammedemin-o4y,"I can't enter Brawl Stars, Supercell, what did...",58 minutes ago,Brawl Stars
4,4,https://www.youtube.com/watch?v=n1gwrRFc4Wc&pp...,@ArasTasuer,Leonnnnnnnnnnnnn. Plssssssssss,1 hour ago,Brawl Stars
...,...,...,...,...,...,...
49302,49302,https://www.youtube.com/watch?v=tm5bWQT-qhk,@pinkpanther9822,12 seconds ago is diabolical,13 days ago,Clash Of Clans
49303,49303,https://www.youtube.com/watch?v=tm5bWQT-qhk,@azrulgym4720,Y,13 days ago,Clash Of Clans
49304,49304,https://www.youtube.com/watch?v=tm5bWQT-qhk,@abhimanyusudheesh,First,13 days ago,Clash Of Clans
49305,49305,https://www.youtube.com/watch?v=tm5bWQT-qhk,@LeeHyZkiee,yow,13 days ago,Clash Of Clans


In [5]:
import re
from nltk.tokenize import word_tokenize
def clean(row):
    row=str(row)
    row=re.sub(r'https\S+|www.\S+',"",row)
    row=re.sub(r"&lt;|&gt;|&amp;", " ", row)
    row=row.strip()
    words=word_tokenize(row)
    if len(words)<3:
        return None
    else:
        return row

data['Text']=data['Text'].apply(clean)
data.dropna(inplace=True)
    


In [6]:
import torch
from transformers import pipeline
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline
model_name_sent='cardiffnlp/twitter-roberta-base-sentiment-latest'
encoder=AutoTokenizer.from_pretrained(model_name_sent)
sentiment_model=AutoModelForSequenceClassification.from_pretrained(
    model_name_sent
    ).to("cpu")

model_name_irony='cardiffnlp/twitter-roberta-base-irony'
irony_tokenizer=AutoTokenizer.from_pretrained(model_name_irony)
model_irony=AutoModelForSequenceClassification.from_pretrained(model_name_irony)
irony_model=pipeline(
    'text-classification',
    model=model_irony,
    tokenizer=irony_tokenizer,
    device=-1
)

c:\ai_env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\ai_env\lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Prabhat\.cache\huggingface\hub\models--cardiffnlp--twitter-roberta-base-sentiment-latest. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this ar

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


In [ ]:
import torch
import torch.nn.functional as F

def get_sentiment_and_score(texts, batch=8):
    results = []
    
    for i in range(0, len(texts), batch):
        batch_texts = list(texts[i:i+batch])
        encoded = encoder(
            batch_texts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=256
        )
        encoded = {k: v.cpu() for k, v in encoded.items()}
        with torch.no_grad():
            outputs = sentiment_model(**encoded)
            logits = outputs.logits
            probs = F.softmax(logits, dim=1)
        pred_ids = torch.argmax(logits, dim=1)
        id2label = {0: "negative", 1: "neutral", 2: "positive"}
        weights = torch.tensor([0.0, 5.0, 10.0])
        ratings = (probs * weights).sum(dim=1)
        sarcasm_scores = []
        for txt in batch_texts:
            out = irony_model(txt, truncation=True, max_length=256)
            sarcasm_scores.append(out[0]["score"])

        sarcasm_scores = torch.tensor(sarcasm_scores)
        ratings = ratings - sarcasm_scores * 1.0
        ratings = torch.clamp(ratings, 0, 10)
        for txt, pid, rating in zip(batch_texts, pred_ids, ratings):
            results.append({
                "Text": txt,
                "Sentiment": id2label[int(pid)],
                "Rating": float(rating)
            })

    return results


In [11]:
temp=get_sentiment_and_score(data['Text'])
temp=pd.DataFrame(temp)
temp

,Text,Sentiment,Rating
0,GIVE OLD BRAWL STAR 🙏🏻,neutral,6.318882
1,1:05 9999 1:09 1:11 1:11 1:11 1:12 1:12 1:12 1...,neutral,4.370735
2,I can’t play the game,negative,1.064428
3,"I can't enter Brawl Stars, Supercell, what did...",negative,0.285543
4,Leonnnnnnnnnnnnn. Plssssssssss,neutral,5.576208
...,...,...,...
41658,"I Suggest, Eagle artirely Merge Clan Casle, TR...",neutral,3.739636
41659,Just started doing videos so I'm just screen r...,positive,7.958956
41660,Let's goooo Im first,positive,8.159514
41661,12 seconds ago is diabolical,negative,0.319776


In [12]:
ID=[]
for i in range(len(data)):
    ID.append(i)
temp['ID']=ID
data['ID']=ID


In [13]:
new_data=pd.merge(data,temp,on='ID',how='inner')
new_data.drop(columns=['Text_x'],inplace=True)
new_data.rename(columns={'Text_y':'text'},inplace=True)


In [14]:
new_data.to_csv("Checkpoint2")

In [15]:
import pandas as pd
import numpy as np
new_data=pd.read_csv(r"C:\Users\Prabhat\Desktop\Carear\projects\Auto snetiment analyser\Checkpoint2")

In [16]:
new_data

,Unnamed: 0.1,Unnamed: 0,url,author,date,category,ID,text,Sentiment,Rating
0,0,0,https://www.youtube.com/watch?v=n1gwrRFc4Wc&pp...,@Ronak-y3z,43 minutes ago,Brawl Stars,0,GIVE OLD BRAWL STAR 🙏🏻,neutral,6.318882
1,1,1,https://www.youtube.com/watch?v=n1gwrRFc4Wc&pp...,@ДамианТертышник,46 minutes ago,Brawl Stars,1,1:05 9999 1:09 1:11 1:11 1:11 1:12 1:12 1:12 1...,neutral,4.370735
2,2,2,https://www.youtube.com/watch?v=n1gwrRFc4Wc&pp...,@CalebKlengenberg,50 minutes ago,Brawl Stars,2,I can’t play the game,negative,1.064428
3,3,3,https://www.youtube.com/watch?v=n1gwrRFc4Wc&pp...,@Muhammedemin-o4y,58 minutes ago,Brawl Stars,3,"I can't enter Brawl Stars, Supercell, what did...",negative,0.285543
4,4,4,https://www.youtube.com/watch?v=n1gwrRFc4Wc&pp...,@ArasTasuer,1 hour ago,Brawl Stars,4,Leonnnnnnnnnnnnn. Plssssssssss,neutral,5.576208
...,...,...,...,...,...,...,...,...,...,...
41658,41658,49297,https://www.youtube.com/watch?v=tm5bWQT-qhk,@sambong25,5 days ago,Clash Of Clans,41658,"I Suggest, Eagle artirely Merge Clan Casle, TR...",neutral,3.739636
41659,41659,49298,https://www.youtube.com/watch?v=tm5bWQT-qhk,@WhiteNinjaCOC,17 hours ago,Clash Of Clans,41659,Just started doing videos so I'm just screen r...,positive,7.958956
41660,41660,49301,https://www.youtube.com/watch?v=tm5bWQT-qhk,@Ryan-e4d,13 days ago,Clash Of Clans,41660,Let's goooo Im first,positive,8.159514
41661,41661,49302,https://www.youtube.com/watch?v=tm5bWQT-qhk,@pinkpanther9822,13 days ago,Clash Of Clans,41661,12 seconds ago is diabolical,negative,0.319776


In [ ]:
from sentence_transformers import SentenceTransformer
import umap # type: ignore
import hdbscan # type: ignore
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np
transformer=SentenceTransformer("all-roberta-large-v1")
def get_topics(df):
    texts=df['text'].astype(str).tolist()
    embeddings=transformer.encode(texts,show_progress_bar=True)
    umap_model=umap.UMAP(n_neighbors=40,n_components=7,metric='cosine')
    umap_embeddings=umap_model.fit_transform(embeddings)
    HDB_model=hdbscan.HDBSCAN(
        min_cluster_size=100,
        min_samples=50,
        metric='euclidean',
        cluster_selection_method='eom'
    )
    cluster_labels=HDB_model.fit_predict(umap_embeddings)
    temp=pd.DataFrame({'text':df['text'],'topics':cluster_labels})
    ans={}
    for topic in temp['topics'].unique():
        all_text=temp[temp['topics']==topic]['text'].tolist()
        all_text=" ".join(all_text)
        ans[topic]=all_text
    ans=pd.DataFrame({
        'topics': ans.keys(),
        'text': ans.values()
        })
    vectorizer=TfidfVectorizer(stop_words='english')
    matrix_tfidf=vectorizer.fit_transform(ans['text'])
    feature_names=vectorizer.get_feature_names_out()
    matrix_tfidf=matrix_tfidf.toarray()
    topic_keywords = {}
    for topic_id, row in zip(ans['topics'], matrix_tfidf):
        top_idx = np.argsort(row)[-20:][::-1]
        top_words = [feature_names[i] for i in top_idx]
        topic_keywords[topic_id] = ", ".join(top_words)
    df['Topic'] = cluster_labels
    mapped = df['Topic'].map(topic_keywords)
    df['Topic_Keywords'] = mapped.where(mapped.notna(), "Misc / Noise")
    return df
    
datasets=[]
for category in new_data['category'].unique():
    datasets.append(get_topics(new_data[new_data['category']==category]))
    
    

c:\ai_env\lib\site-packages\huggingface_hub\file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Batches: 100%|██████████| 356/356 [1:23:49<00:00, 14.13s/it]
c:\ai_env\lib\site-packages\sklearn\utils\deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
c:\ai_env\lib\site-packages\sklearn\utils\deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
C:\Users\Prabhat\AppData\Local\Temp\ipykernel_21896\1313583889.py:38: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/

In [ ]:
for i,category in enumerate(new_data['category'].unique()):
    table=datasets[i]
    table=pd.DataFrame(table)
    table.to_csv(f"{category}_data.csv") # checkpoint 3

In [1]:
datasets=[]
import pandas as pd
datasets.append(pd.read_csv(r"C:\Users\Prabhat\Desktop\Carear\projects\Auto snetiment analyser\data\Clash Of Clans_data.csv"))
datasets.append(pd.read_csv(r"C:\Users\Prabhat\Desktop\Carear\projects\Auto snetiment analyser\data\Clash Royale_data.csv"))
datasets.append(pd.read_csv(r"C:\Users\Prabhat\Desktop\Carear\projects\Auto snetiment analyser\data\Brawl Stars_data.csv"))

In [2]:
final_data=datasets[0]
final_data=pd.DataFrame(final_data)
for i in range(len(datasets)-1):
    temp=datasets[i+1]
    temp=pd.DataFrame(temp)
    print(final_data['category'].unique())
    print(final_data['Topic'].nunique())
    final_data=pd.concat([final_data,temp],ignore_index=True)
final_data.drop(columns=['Unnamed: 0.1', 'Unnamed: 0'],inplace=True)
final_data.rename(columns={"Topic":"topic_id"},inplace=True)
final_data.to_csv("Final_data.csv")



['Clash Of Clans']
24
['Clash Of Clans' 'Clash Royale']
33


In [3]:
import os
import time
import requests
import pandas as pd

HF_TOKEN = os.getenv("HF_TOKEN")  
API_URL = "https://router.huggingface.co/v1/chat/completions"

headers = {
    "Authorization": f"Bearer {HF_TOKEN}",
    "Content-Type": "application/json"
}
names = []
def query(prompt, retries=30):
    for attempt in range(retries):
        payload = {
            "model": "Qwen/Qwen2.5-72B-Instruct",
            "provider":'hf-inference',
            "messages": [
                {"role": "user", "content": prompt}
            ],
            "max_tokens": 2000,
            "temperature": 0.3
        }
        resp = requests.post(API_URL, headers=headers, json=payload)
        response_data = resp.json()
        # Model not ready = try again
        if "error" in response_data:
            print(f"Retry {attempt+1}/{retries} - {response_data['error']}")
            time.sleep(6)
            continue
        # Valid response
        return response_data["choices"][0]["message"]["content"]

    raise Exception("Model did not load or too many retries")
for data in datasets:
    data = pd.DataFrame(data)

    for id in data['Topic'].unique():
        category = data.loc[data['Topic'] == id, 'category'].iloc[0]
        keywords = data.loc[data['Topic'] == id, 'Topic_Keywords'].unique().tolist()
        sample_texts = data.loc[data['Topic'] == id, 'text'].sample(
            n=min(10, len(data.loc[data['Topic'] == id]))
        ).tolist()

        prompt = f"""
You are an expert topic naming assistant.

Your task:
Given a category, a list of cluster keywords, and sample comments,
generate a SHORT business-friendly topic name (max 4-6 words).

Rules:
- Use BOTH the keywords and sample texts to understand the topic.
- The name must be specific and meaningful.
- Do NOT include the category name inside the topic name.
- Do NOT create vague names like "General Issues" or "Miscellaneous".
- Do NOT explain anything. Only output JSON.

STRICT JSON RULES:
- Output ONLY valid JSON.
- No markdown. No backticks. No text outside the JSON.
- Must start with '{' and end with '}'.

Output format:
{{
  "topic_id": "{id}",
  "category_name": "{category}",
  "topic_name": "<<your_topic_here>>"
}}

Inputs:
category: {category}
keywords: {keywords}
sample_texts: {sample_texts[:3]}
"""


        result = query(prompt)
        names.append(result)

print("\nFirst Output Sample:\n", names[0])
print("\nTotal generated:", len(names))


Retry 1/30 - You have reached the free monthly usage limit for novita. Subscribe to PRO to get 20x more included usage, or add pre-paid credits to your account.
Retry 2/30 - You have reached the free monthly usage limit for novita. Subscribe to PRO to get 20x more included usage, or add pre-paid credits to your account.
Retry 3/30 - You have reached the free monthly usage limit for novita. Subscribe to PRO to get 20x more included usage, or add pre-paid credits to your account.
Retry 4/30 - You have reached the free monthly usage limit for novita. Subscribe to PRO to get 20x more included usage, or add pre-paid credits to your account.
Retry 5/30 - You have reached the free monthly usage limit for novita. Subscribe to PRO to get 20x more included usage, or add pre-paid credits to your account.
Retry 6/30 - You have reached the free monthly usage limit for novita. Subscribe to PRO to get 20x more included usage, or add pre-paid credits to your account.
Retry 1/30 - You have reached the 

In [4]:
print(names)

['{\n  "topic_id": "-1",\n  "category_name": "Clash Of Clans",\n  "topic_name": "Town Hall 18 Updates and Features"\n}', '{\n  "topic_id": "16",\n  "category_name": "Clash Of Clans",\n  "topic_name": "TH17 vs TH18 Design"\n}', '{\n  "topic_id": "0",\n  "category_name": "Clash Of Clans",\n  "topic_name": "Eric Ban Controversy"\n}', '{\n  "topic_id": "19",\n  "category_name": "Clash Of Clans",\n  "topic_name": "New Spell Mechanics and Impact"\n}', '{\n  "topic_id": "10",\n  "category_name": "Clash Of Clans",\n  "topic_name": "Town Hall Design and Levels"\n}', '{\n  "topic_id": "3",\n  "category_name": "Clash Of Clans",\n  "topic_name": "Voice Acting and Characters"\n}', '{\n  "topic_id": "13",\n  "category_name": "Clash Of Clans",\n  "topic_name": "Judo Sloth Content & Updates"\n}', '{\n  "topic_id": "2",\n  "category_name": "Clash Of Clans",\n  "topic_name": "Gameplay Issues and Updates"\n}', '{\n  "topic_id": "11",\n  "category_name": "Clash Of Clans",\n  "topic_name": "Player Sentimen

In [4]:
import json
parsed = []
for x in names:
    parsed.append(json.loads(x))
names=pd.DataFrame(parsed)
names

,topic_id,category_name,topic_name
0,-1,Clash Of Clans,TH18 Update and Aesthetics
1,16,Clash Of Clans,TH Design Evolution
2,0,Clash Of Clans,Eric Banning Controversy
3,19,Clash Of Clans,New Spell Impact
4,10,Clash Of Clans,Town Hall Design Critiques
...,...,...,...
58,2,Brawl Stars,Character Requests and Preferences
59,4,Brawl Stars,Event Rewards and Bugs
60,3,Brawl Stars,Game Updates and Features
61,1,Brawl Stars,Event Accessibility Issues


In [5]:
names['topic_name'].nunique()

61

In [ ]:
import os
import time
import json
import requests
import pandas as pd

API_URL = "https://router.huggingface.co/v1/chat/completions"
headers = {
    "Authorization": f"Bearer {os.getenv('HF_TOKEN')}",
    "Content-Type": "application/json"
}

subtopics = []


def query_generalization(category, topics, retries=20):
    prompt = f"""
You will be given a list of topic names for a specific category.

Your job:
- If the topics are too specific, redundant, or overlap → generalize them into broader business-friendly topics.
- If the topics are already broad and distinct → return an empty JSON list [].

Important rules:
- Output must be only a VALID JSON ARRAY.
- No text outside the JSON.
- Each item must contain:
    "category_name": "{category}"
    "old": "<original_old_topic>"
    "new": "<new_generalized_topic>"
- If nothing to generalize → output: []

Output format example:
[
  {{
    "category_name": "{category}",
    "old": "original topic",
    "new": "new generalized topic"
  }}
]

Now here are the topics to analyze:
{topics}
"""
    for attempt in range(retries):
        payload = {
            "model": "Qwen/Qwen2.5-72B-Instruct",
            "provider": "hf-inference",
            "messages": [
                {"role": "user", "content": prompt}
            ],
            "max_tokens": 2000,
            "temperature": 0.2
        }

        resp = requests.post(API_URL, headers=headers, json=payload)
        data = resp.json()

        if "error" in data:
            print(f"[{category}] Retry {attempt+1}/{retries} - {data['error']}")
            time.sleep(4)
            continue

        return data["choices"][0]["message"]["content"]

    return "[]" 

for category in names['category_name'].unique():
    topics = names[names['category_name'] == category]['topic_name'].unique().tolist()
    
    result = query_generalization(category, topics)
    subtopics.append(result)
    
clean_subtopics = []

for block in subtopics:
    try:
        clean_block = block.strip()
        clean_block = clean_block.replace("```json", "").replace("```", "").strip()
        parsed = json.loads(clean_block)
        if isinstance(parsed, list):
            clean_subtopics.extend(parsed)
    except json.JSONDecodeError:
        print("\n⚠️ Bad JSON skipped:\n", block[:200])

generalized_df = pd.DataFrame(clean_subtopics)

print("\nFinal Sample Output:")
print(generalized_df.head())
print("\nTotal Generalized Rows:", len(generalized_df))



Processing category: Clash Of Clans

Processing category: Clash Royale

Processing category: Brawl Stars

Final Sample Output:
    category_name                         old  \
0  Clash Of Clans  TH18 Update and Aesthetics   
1  Clash Of Clans         TH Design Evolution   
2  Clash Of Clans  Town Hall Design Critiques   
3  Clash Of Clans      Wall Design Aesthetics   
4  Clash Of Clans        Builder Base Updates   

                                new  
0  Town Hall Updates and Aesthetics  
1  Town Hall Updates and Aesthetics  
2  Town Hall Updates and Aesthetics  
3  Town Hall Updates and Aesthetics  
4     Base Updates and Improvements  

Total Generalized Rows: 58


In [7]:
subtopics=generalized_df


In [8]:
final_data['topic_id'] = final_data['topic_id'].astype(int)
names['topic_id'] = names['topic_id'].astype(int)
merged = final_data.merge(
    names,
    how="left",
    left_on=["category", "topic_id"],
    right_on=["category_name", "topic_id"]
)
merged = merged.merge(
    subtopics,
    how="left",
    left_on=["category_name", "topic_name"],
    right_on=["category_name", "old"]
)
merged['Topic'] = merged['new'].fillna(merged['topic_name'])
merged['Subtopic'] = None          
mask = merged['new'].notna()          
merged.loc[mask, 'Subtopic'] = merged.loc[mask, 'topic_name']
merged = merged.drop(columns=['old', 'new', 'category_name'])


In [ ]:

import pandas as pd
from datetime import datetime, timedelta
import re

scraped_date = datetime(2025, 12, 4, 20, 25)

def convert_relative_time(value):
    value = value.lower()
    value = (
        value.replace("\xa0", " ")
             .replace("(edited)", "")
             .replace("edited", "")
             .replace("ago", "")
             .replace("•", "")
             .strip()
    )
    if value == "" or value == "just now":
        return scraped_date
    if "yesterday" in value:
        return scraped_date - timedelta(days=1)
    match = re.search(r"(\d+)", value)
    num = int(match.group()) if match else None
    if num is not None and "second" in value:
        return scraped_date - timedelta(seconds=num)
    if num is not None and "minute" in value:
        return scraped_date - timedelta(minutes=num)
    if num is not None and "hour" in value:
        return scraped_date - timedelta(hours=num)
    if num is not None and "day" in value:  
        return scraped_date - timedelta(days=num)
    if num is not None and "week" in value:
        return scraped_date - timedelta(weeks=num)
    if num is not None and "month" in value:
        return scraped_date - pd.DateOffset(months=num)
    if num is not None and "year" in value:
        return scraped_date - pd.DateOffset(years=num)
    try:
        return pd.to_datetime(value)
    except:
        return None
merged['parsed_date'] = merged['date'].apply(convert_relative_time)
merged.drop(columns=['topic_name'],inplace=True)


In [ ]:
merged.drop_duplicates(inplace=True)

In [ ]:
for category in merged['category'].unique():
    print(merged[merged['category']==category]['Topic'].nunique())

In [29]:
merged.to_csv("Data_For_Analysis")

In [17]:
merged=pd.read_csv(r"C:\Users\Prabhat\Desktop\Carear\projects\Auto snetiment analyser\data\Data_For_Analysis")